# Random Forest — Bars and Stripes

Misma estructura que `RandomForest_Iris`, datos PennyLane Bars and Stripes.


## Librerias


In [ ]:
# 1. Import all libraries
import json
import time
import urllib.request
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from pennylane.data import Dataset
from sklearn.preprocessing import MinMaxScaler, label_binarize
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

np.random.seed(123)


## Data

Imágenes oficiales PennyLane (`4×4` → 16 features). Carga lazy del `.h5`.


In [ ]:
DATASET_URL = (
    "https://datasets.cloud.pennylane.ai/datasets/h5/"
    "other/BarsAndStripes/BarsAndStripes_BarsAndStripes.h5"
)
IMAGE_SIZE = "4"  # 4x4 pixels -> 16 features


def resolve_dataset_path() -> Path:
    """Busca el .h5 en caches locales tipicos de PennyLane."""
    candidates = []
    for base in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        candidates.append(
            base / "datasets" / "other" / "BarsAndStripes" / "BarsAndStripes_BarsAndStripes.h5"
        )
    for path in candidates:
        if path.is_file():
            return path
    return candidates[0]


def ensure_bars_and_stripes(path: Path | None = None) -> Path:
    path = resolve_dataset_path() if path is None else Path(path)
    if path.is_file():
        return path
    path.parent.mkdir(parents=True, exist_ok=True)
    print(f"Downloading PennyLane bars-and-stripes -> {path} (one-time, ~750MB)")
    urllib.request.urlretrieve(DATASET_URL, path)
    return path


def load_bars_and_stripes(image_size: str = IMAGE_SIZE):
    path = ensure_bars_and_stripes()
    ds = Dataset.open(path, "r")
    try:
        X_train = np.asarray(ds.train[image_size]["inputs"], dtype=float)
        y_train = np.asarray(ds.train[image_size]["labels"], dtype=float)
        X_test = np.asarray(ds.test[image_size]["inputs"], dtype=float)
        y_test = np.asarray(ds.test[image_size]["labels"], dtype=float)
    finally:
        ds.close()
    return X_train, y_train, X_test, y_test


print("Loading PennyLane 'bars-and-stripes' (image_size=4)...")
X_train_official, y_train_official, X_test_official, y_test_official = load_bars_and_stripes(IMAGE_SIZE)
print(
    f"Official splits -> train: {X_train_official.shape}, test: {X_test_official.shape}"
)
print("Label values (train):", np.unique(y_train_official, return_counts=True))


In [ ]:
# Preview: first samples as 4x4 images + label
fig, axs = plt.subplots(1, 4, figsize=(10, 3), dpi=120)
for i, ax in enumerate(axs):
    ax.imshow(X_train_official[i].reshape(4, 4), cmap="gray")
    ax.set_title(f"y={int(y_train_official[i])}")
    ax.axis("off")
fig.suptitle("Bars and Stripes — sample train images (4×4)", color="#0E4A75")
plt.tight_layout()
plt.show()


## Data Preparation


In [ ]:
# Combine official splits, map labels {-1,1}->{0,1}, Iris protocol
X = np.concatenate([X_train_official, X_test_official], axis=0)
y = np.concatenate([y_train_official, y_test_official], axis=0)
y = np.where(y <= 0, 0, 1).astype(int)

n_features = X.shape[1]
print(f"Combined dataset: n={len(X)}, features={n_features}, classes={np.unique(y, return_counts=True)}")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.4, random_state=123, stratify=y
)
print(f"Custom Dataset Loaded: Train={len(X_train)}, Test={len(X_test)}")

minmax = MinMaxScaler(feature_range=(0, np.pi))
X_train = minmax.fit_transform(X_train)
X_test = minmax.transform(X_test)
X_test = np.clip(X_test, 0, np.pi)
print(f"Final shapes -> Train: {X_train.shape}, Test: {X_test.shape}")


## Random Forest


In [ ]:
class RandomForestModel:
    def __init__(self):
        self.model = RandomForestClassifier(
            n_estimators=100,
            random_state=123,
        )

    def fit(self, X, y):
        self.model.fit(X, y)
        return self

    def predict(self, X):
        return self.model.predict(X)

    def predict_proba(self, X):
        return self.model.predict_proba(X)


## Entrenamiento Random Forest


In [ ]:
# Training and timing statistics
print("=== Train Random Forest ===")
rf_model = RandomForestModel()
start = time.time()
rf_model.fit(X_train, y_train)
rf_time = time.time() - start
print(f"Random Forest training time: {rf_time:.2f}s")


## Model Evaluation


In [ ]:
def evaluate(model, X_t, y_t, classes):
    preds = model.predict(X_t)
    scores = model.predict_proba(X_t)

    acc = accuracy_score(y_t, preds)
    prec = precision_score(y_t, preds, average="macro", zero_division=0)
    rec = recall_score(y_t, preds, average="macro", zero_division=0)
    f1 = f1_score(y_t, preds, average="macro", zero_division=0)
    if scores.shape[1] == 2:
        auc = roc_auc_score(y_t, scores[:, 1])
    else:
        auc = roc_auc_score(label_binarize(y_t, classes=classes), scores, average="macro")
    return acc, prec, rec, f1, auc, scores, preds


In [ ]:
classes = np.unique(y_train)

rf_acc, rf_prec, rf_rec, rf_f1, rf_auc, rf_probs, rf_preds = evaluate(
    rf_model, X_test, y_test, classes
)
rf_acc_tr, rf_prec_tr, rf_rec_tr, rf_f1_tr, rf_auc_tr, _, _ = evaluate(
    rf_model, X_train, y_train, classes
)

print("\n=== Train vs Test ===")
print(
    f"RF Train - Acc: {rf_acc_tr:.4f} | Prec: {rf_prec_tr:.4f} | Rec: {rf_rec_tr:.4f} | "
    f"F1: {rf_f1_tr:.4f} | AUC: {rf_auc_tr:.4f}"
)
print(
    f"RF Test  - Acc: {rf_acc:.4f} | Prec: {rf_prec:.4f} | Rec: {rf_rec:.4f} | "
    f"F1: {rf_f1:.4f} | AUC: {rf_auc:.4f}"
)


In [ ]:
# Result Visualization (train vs test + paleta UNIR)
plt.rcParams.update({
    "font.sans-serif": ["Arial"],
    "axes.edgecolor": "#C0C0C8",
    "grid.color": "#C0C0C8",
    "grid.linestyle": "--",
})

def get_inference_time(model, data):
    start = time.time()
    _ = model.predict(data)
    return time.time() - start

rf_inf_time = get_inference_time(rf_model, X_test)

train_color = "#0E4A75"
test_color = "#219BC3"
unir_gris = "#858591"

fig, axs = plt.subplots(1, 2, figsize=(14, 5), dpi=150)
fig.suptitle("Random Forest — Bars and Stripes", fontsize=18, y=1.02, color="#0E4A75")

metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'AUC-ROC']
train_vals = [rf_acc_tr, rf_prec_tr, rf_rec_tr, rf_f1_tr, rf_auc_tr]
test_vals = [rf_acc, rf_prec, rf_rec, rf_f1, rf_auc]
x = np.arange(len(metrics))
width = 0.35

for i in range(len(metrics)):
    axs[0].bar(x[i] - width / 2, train_vals[i], width, color=train_color,
               label="Train" if i == 0 else "", zorder=3)
    axs[0].bar(x[i] + width / 2, test_vals[i], width, color=test_color,
               label="Test" if i == 0 else "", zorder=3)

axs[0].set_ylabel("Score")
axs[0].set_title("Train vs Test Metrics", color="#0E4A75")
axs[0].set_xticks(x)
axs[0].set_xticklabels(metrics)
axs[0].set_ylim(0, 1.35)
axs[0].legend(loc="upper right")
axs[0].grid(axis="y", zorder=0)

eff_labels = ["Training Time", "Inference Time"]
eff = [rf_time, rf_inf_time]
x_eff = np.arange(len(eff_labels))
axs[1].bar(x_eff, eff, 0.5, color="#0E4A75", label="Random Forest", zorder=3)
axs[1].set_xticks(x_eff)
axs[1].set_xticklabels(eff_labels)
axs[1].set_ylabel("Time (s)")
axs[1].set_title("Computational Efficiency", color="#0E4A75")
max_time = max(eff)
axs[1].set_ylim(0, max_time * 1.35 if max_time > 0 else 1)
axs[1].legend(loc="upper right")
axs[1].grid(axis="y", zorder=0)
offset = max_time * 0.02 if max_time > 0 else 0.02
for i in range(2):
    axs[1].text(x_eff[i], eff[i] + offset, f"{eff[i]:.4f}s",
                ha="center", va="bottom", fontsize=9, color=unir_gris)

plt.tight_layout()
plt.show()


## Guardar resultados


In [ ]:
# Save metrics for QuantumClassification comparison
def resolve_metrics_dir() -> Path:
    """Guarda en Notebooks/QuantumClassification/metrics."""
    for base in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        quantum = base / "Notebooks" / "QuantumClassification"
        if quantum.is_dir():
            metrics_dir = quantum / "metrics"
            metrics_dir.mkdir(parents=True, exist_ok=True)
            return metrics_dir
    metrics_dir = Path.cwd().resolve() / "Notebooks" / "QuantumClassification" / "metrics"
    metrics_dir.mkdir(parents=True, exist_ok=True)
    return metrics_dir

metrics_dir = resolve_metrics_dir()

payload = {
    "model": "Random Forest",
    "family": "classical",
    "protocol": {
        "dataset": "bars-and-stripes",
        "image_size": int(IMAGE_SIZE),
        "n_features": int(n_features),
        "test_size": 0.4,
        "random_state": 123,
        "stratify": True,
        "scaler": "MinMaxScaler[0, pi]",
        "labels": "mapped {-1,1}->{0,1}",
        "pca": None,
    },
    "train": {
        "accuracy": float(rf_acc_tr),
        "precision": float(rf_prec_tr),
        "recall": float(rf_rec_tr),
        "f1": float(rf_f1_tr),
        "auc": float(rf_auc_tr),
    },
    "test": {
        "accuracy": float(rf_acc),
        "precision": float(rf_prec),
        "recall": float(rf_rec),
        "f1": float(rf_f1),
        "auc": float(rf_auc),
    },
    "train_time_s": float(rf_time),
    "inference_time_s": float(rf_inf_time),
}

out = metrics_dir / "rf_bs.json"
out.write_text(json.dumps(payload, indent=2))
print("Saved:", out)
